# DCAT-AP-NL Requirement Analysis for Dataverse Metadata Exporter



In [177]:
# boiler plate functions
# imports SPARQL prefixes and functions defs
import csv
from pprint import pprint
# from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, CSV 
from rdflib import Graph

prefixes = '''    
PREFIX adms: <http://www.w3.org/ns/adms#>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcatap: <http://data.europa.eu/r5r/>
PREFIX eli: <http://data.europa.eu/eli/ontology#>
PREFIX eush: <https://purl.eu/ns/shacl#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX vcard: <http://www.w3.org/2006/vcard/ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

PREFIX dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/>
'''    

def sparql_files_query(query, format, filespath):
    g = Graph()
    for filepath in filespath:
        g.parse(filepath, format=format)  # can also use "ttl" for Turtle
    query = prefixes + query
    results = g.query(query)
    return results

def create_csv(filepath, headers, data_dict):

    with open(filepath, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data_dict)


# def sparql_query(query, format):
#     formats = {"json": JSON, "turtle": TURTLE, "csv": CSV}
#     f_ = formats[format]
#     endpoint = "http://vocab.getty.edu/sparql"
#     sparql = SPARQLWrapper(endpoint)
#     query = prefixes + query     
#     sparql.setQuery(query)
#     sparql.setReturnFormat(f_)
#     results = sparql.query().convert()    
#     return results


# def print_sparql_results(results):
#     for row in results["results"]["bindings"]:
#         return (row)


# Requirement: Mandatory Dataset properties - Research

**What the mandatory properties of dcat:Dataset in DCAT-AP?**
ie. `dct:description`

**What the mandatory properties of dcat:Dataset in DCAT-AP-NL?**
ie. `dct:identifier`

* include cardinality and property range


According to https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#17C1E0BE
> The SHACL rules of DCAT-AP-NL build on the SHACL rules from DCAT API-3.0. **All the rules from DCAT-AP-3.0 (see [DCAT AP-3.0 dcat-ap-SHACL.ttl](dcat-ap/releases/3.0.0/shacl/dcat-ap-SHACL.ttl)) are still applicable. DCAT-AP-NL only tightens some data rules.**

>To test whether a dataset description meets DCAT-AP-NL, it is also necessary to include both the DCAT-AP SHACL shapes and the DCAT-AP-NL SHACL shapes in the validation.

> To properly support the validation of the dataset descriptions, DCAT-AP breaks the SHACL shapes is split to support different validation scenarios and aspects. See the chapter **[Validation of DCAT-AP](https://semiceu.github.io/DCAT-AP/releases/3.0.0/#validation-of-dcat-ap)**.

**DCAT-AP-NL SHACL shapes** are divided into:

* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl](dcat-ap-nl-SHACL.ttl): The SHACL shapes of DCAT-AP-NL, excluding the validation rules around the class range of properties.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik.ttl](dcat-ap-nl-SHACL-klassebereik.ttl) The SHACL shapes of DCAT-AP-EN for validating the class range of properties, excluding the class range of properties with a value derived from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl](dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl:) dcat-ap-nl-SHACL class range-codelists.ttl : The SHACL shapes of DCAT-AP-EN for validating the class range of properties with a value from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl](dcat-ap-nl-SHACL-aanbevolen.ttl): The SHACL shapes of DCAT-AP-EN for validating recommended properties.



In [178]:
# INVESTIGATION
# Goal: understand how the cardinality 1..(mandatory) is expressed in DCAT-AP shacl
# By: SPARQL DESCRIBE of dcat-ap-SHACL.ttl dcat:Dataset: dct:description  shape in DCAT-AP SHACL
# Answer: via property:value  shacl:minCount 1 ;

sparql_dcatap_dataset_1mandatory_props = ''' 
DESCRIBE ?prop_shape
WHERE {
    BIND(dct:description AS ?prop_path) .
    <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
    ?prop_shape sh:path ?prop_path .
}
'''
results = sparql_files_query(query=sparql_dcatap_dataset_1mandatory_props, 
                            format='ttl', 
                            filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])
print(results.serialize(format='ttl').decode('utf-8'))


@prefix dc1: <http://purl.org/dc/terms/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix shacl: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/6cae9880e515253132af1452a38a8a5827165149> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.description" ;
    shacl:description "A free-text account of the Dataset."@en ;
    shacl:name "description"@en ;
    shacl:nodeKind shacl:Literal ;
    shacl:path dc1:description .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.description" ;
    shacl:description "A free-text account of the Dataset."@en ;
    shacl:minCount 1 ;
    shacl:name "description"@en ;
    shacl:path dc1:description .




The pattern I see in the cell above (`dcat:Dataset: dct:description`) property shapes, makes me conclude that 
* in **DCAT-AP SHACL the required properties have `shacl:minCount 1`**, which makes sense

Follow-up questions/queries?

* which other Dataset properties have `shacl:minCount 1` AKA are mandatory? 
* is the same pattern present in DCAT-AP-NL shacl?

In [179]:
# OUTPUT
# Goal: list of all dcat:Dataset mandatory properties in DCAT-AP & DCAT-APN-NL
# shacl:minCount 1

sparql_apnl_dataset_mandatory_props = ''' 
SELECT ?prop_path  ?prop_shape 
WHERE {
    {   # DCAT-AP query
        <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
        ?prop_shape sh:minCount 1 ;
            sh:path ?prop_path .
    }
    UNION
    {   # DCAT-AP-NL query
        dcatapnl-sh:DatasetShape sh:property ?prop_shape .
        ?prop_shape sh:minCount 1 ;
            sh:path ?prop_path . 
    }
}
'''


results_ap_nl = sparql_files_query(query=sparql_apnl_dataset_mandatory_props, 
                            format='ttl', 
                            filespath=[ 
                                'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl',
                                'dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])

print(f"{'*'*10} Dataset Mandatory Properties in DCAT-AP & DCAT-AP-NL  {'*'*10}")
for row in results_ap_nl:
    pprint(row.asdict())

ap_NL_mandatory_dataset_props_list = [row.asdict() for row in results_ap_nl]

# create_csv(filepath='dcat-ap-nl_mand_props.csv',
#            headers=ap_NL_mandatory_dataset_props_list[0].keys(),
#            data_dict=ap_NL_mandatory_dataset_props_list)


********** Dataset Mandatory Properties in DCAT-AP & DCAT-AP-NL  **********
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/description'),
 'prop_shape': rdflib.term.URIRef('https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6')}
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/title'),
 'prop_shape': rdflib.term.URIRef('https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/f8b02efd063b8089b72ce9677a1e4a3488eeb9a9')}
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/accessRights'),
 'prop_shape': rdflib.term.URIRef('http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_accessRights_minCount')}
{'prop_path': rdflib.term.URIRef('http://www.w3.org/ns/dcat#contactPoint'),
 'prop_shape': rdflib.term.URIRef('http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_contactPoint_minCount')}
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/creator'),
 'prop_shape': 

# Requirement: Mandatory Dataset properties

The response to the query above, tells us that the mandatory Dataset properties are:

See [ap-nl-dataset-overview.csv](ap-nl-dataset-overview.csv) where this info is compiled

**DCAT-AP mandatory properties of dcat:Dataset:**

*  http://purl.org/dc/terms/description
*  http://purl.org/dc/terms/title 

**DCAT-AP-NL mandatory properties of dcat:Dataset:**

* http://purl.org/dc/terms/accessRights 
* http://www.w3.org/ns/dcat#contactPoint 
* http://purl.org/dc/terms/creator 
* http://purl.org/dc/terms/identifier 
* http://purl.org/dc/terms/publisher 
* http://www.w3.org/ns/dcat#theme


# Requirement Enunciation: **Range** of DCAT-AP + DCAT-AP-NL Mandatory Properties

The range is the type of values a property can have.

The focus here is to **find the ranges of *object properties* (that have other RDF nodes as their value)** in n [dcat-ap/releases/3.0.1/shacl/ranges.ttl](dcat-ap/releases/3.0.1/shacl/ranges.ttl). *Data properties*, that have strings or numbers as values, are not being address by ranges.ttl.


Example of Dataset dcat:creator property and its range description defined by `shacl:class foaf:Agent`

```
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/c1d40f7102c8201949576e76be48b991b47958d9> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.creator";
  shacl:class foaf:Agent;
  shacl:description "An entity responsible for producing the dataset."@en;
  shacl:name "creator"@en;
  shacl:path dc:creator .
```




In [180]:
# Goal: info **Ranges** of Dataset mandatory object properties
# note: object properties have as values RDF nodes (class instances), in contrast with data properties which have literals as values
# Output: Dataset mandatory object properties shapes - 
# property in schal:path; range in shacl:class
#  

prop_path_for_sparql = [item['prop_path'] for item in ap_NL_mandatory_dataset_props_list]
prop_path_for_sparql_str4query = (' '.join([f'<{str(uri)}>' for uri in prop_path_for_sparql]))
# use prop_path_for_sparql_str4query as VALUES in following range query
sparql_dcat_mandatory_dataset_props_ranges = '''
DESCRIBE  ?prop_shape_uri
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path  .
    
}''' % prop_path_for_sparql_str4query
print(f"{'*'*3} Querying shapes for Dataset mandatory propreties: {prop_path_for_sparql_str4query} {'*'*3}\n")
describe_dcatap_NL_mandatory_dataset_props = sparql_files_query(query=sparql_dcat_mandatory_dataset_props_ranges, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
print(describe_dcatap_NL_mandatory_dataset_props.serialize(format='ttl').decode('utf-8'))

# TODO: include this info in the CSV dcat-ap-nl_mand_props.csv

*** Querying shapes for Dataset mandatory propreties: <http://purl.org/dc/terms/description> <http://purl.org/dc/terms/title> <http://purl.org/dc/terms/accessRights> <http://www.w3.org/ns/dcat#contactPoint> <http://purl.org/dc/terms/creator> <http://purl.org/dc/terms/identifier> <http://purl.org/dc/terms/publisher> <http://www.w3.org/ns/dcat#theme> ***

@prefix dc1: <http://purl.org/dc/terms/> .
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix shacl: <http://www.w3.org/ns/shacl#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix vcard: <http://www.w3.org/2006/vcard/ns#> .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/7b6713c1f4a52e964f5db57eabef294b6d04e90e> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.contactpoint" ;
    shacl:class vcard:Kind ;
    shacl:description "Contact information that can be used for sending co

In [181]:
# continuation of previous cell 
# Output:  for DCAT-AP-NL Dataset mandatory properties and their range (sh:class ?prop_range) 
# Output: ap-nl-dataset-overview.csv

print(f'{"-"*40}\nQuerying in dcat-ap/releases/3.0.1/shacl/ranges.ttl Dataset property shapes:\n{prop_path_for_sparql_str4query}\n{"-"*40}')

sparql_dcat_mandatory_dataset_props_ranges_vars = '''
SELECT  ?prop_path  ?prop_range
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path ;
                    sh:class ?prop_range .
    
}''' % prop_path_for_sparql_str4query


ap_NL_mandatory_dataset_props_range = sparql_files_query(query=sparql_dcat_mandatory_dataset_props_ranges_vars, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
ap_NL_mandatory_dataset_props_range_list = [row.asdict() for row in ap_NL_mandatory_dataset_props_range]

# Join  ap_NL_mandatory_dataset_props_range_list & ap_NL_mandatory_dataset_props_list
# Step 1: Build a mapping from prop_path to prop_range
prop_range_map = {d['prop_path']: d['prop_range'] for d in ap_NL_mandatory_dataset_props_range_list}
# Step 2: Merge the lists
ap_NL_mandatory_dataset_props_merged = []
for d in ap_NL_mandatory_dataset_props_list:
    # Copy to avoid mutating the original
    merged_dict = d.copy()
    prop_path = d['prop_path']
    if prop_path in prop_range_map:
        merged_dict['prop_range'] = prop_range_map[prop_path]
    ap_NL_mandatory_dataset_props_merged.append(merged_dict)
print(f"{'*'*3} Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-overview.csv {'*'*3}\n")
# print(ap_NL_mandatory_dataset_props_merged)
create_csv(filepath='ap-nl-dataset-overview.csv',
           headers=['prop_path', 'prop_range', 'prop_shape'],
           data_dict=ap_NL_mandatory_dataset_props_merged)


----------------------------------------
Querying in dcat-ap/releases/3.0.1/shacl/ranges.ttl Dataset property shapes:
<http://purl.org/dc/terms/description> <http://purl.org/dc/terms/title> <http://purl.org/dc/terms/accessRights> <http://www.w3.org/ns/dcat#contactPoint> <http://purl.org/dc/terms/creator> <http://purl.org/dc/terms/identifier> <http://purl.org/dc/terms/publisher> <http://www.w3.org/ns/dcat#theme>
----------------------------------------
*** Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-overview.csv ***



In [186]:
%%!
echo "---- DCAT-AP-NL - mandatory props range ------"
csvtk pretty ap-nl-dataset-overview.csv


['---- DCAT-AP-NL - mandatory props range ------',
 'prop_path                                prop_range                                    prop_shape                                                                                                 ',
 '--------------------------------------   -------------------------------------------   -----------------------------------------------------------------------------------------------------------',
 'http://purl.org/dc/terms/description                                                   https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6',
 'http://purl.org/dc/terms/title                                                         https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/f8b02efd063b8089b72ce9677a1e4a3488eeb9a9',
 'http://purl.org/dc/terms/accessRights    http://purl.org/dc/terms/RightsStatement      http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetSha

# Controlled Vocabularies Constraints

[dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl](dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl) specifies the controlled vocabulary constraints on properties expressed by DCAT-AP in SHACL.

More info in [DCAT-AP Documentation on CVs](https://semiceu.github.io/DCAT-AP/releases/3.0.1/#controlled-vocabularies-to-be-used)

## Data theme CV
From the mandatory Dataset properties, only dcat:theme has a skos:Concept as range. 
The advised vocabulary to use is the "Data theme" http://publications.europa.eu/resource/authority/data-theme (view [human-readable interface](https://op.europa.eu/web/eu-vocabularies/concept-scheme/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme)). Where the candidate concept for SSH/ODISSEI seems to be :
* [SOCI](http://publications.europa.eu/resource/authority/data-theme/SOCI) Population and society

Other possible themes are [ECON](https://op.europa.eu/en/web/eu-vocabularies/concept/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme/ECON) Economy and finance & [EDUC](https://op.europa.eu/web/eu-vocabularies/concept/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme/EDUC) Education, culture and sport, but these seem to be too specific and not matching the SSH DS and ODISSEI Portal domains. https://op.europa.eu/en/web/eu-vocabularies/concept-scheme/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme contains more detailed info 




Overview of DataThemeRestriction in [ap-nl-dataset-CVs.csv](ap-nl-dataset-CVs.csv)

```
:DataThemeRestriction
    a sh:NodeShape ;
    rdfs:comment "Data Theme Restriction" ;
    rdfs:label "Data Theme Restriction" ;
    sh:property [
        sh:hasValue <http://publications.europa.eu/resource/authority/data-theme> ;
        sh:minCount 1 ;
        sh:nodeKind sh:IRI ;
        sh:path skos:inScheme
    ] .

:Dataset_ShapeCV
    a sh:NodeShape ;
    sh:property [
        sh:node :DataThemeRestriction ;
        sh:nodeKind sh:IRI ;
        sh:path dcat:theme ;
	sh:description "Multiple themes can be used but at least one concept of <http://publications.europa.eu/resource/authority/data-theme> should be present" ;
        sh:severity sh:Warning
    ], ...    
```

* start query at :Dataset_ShapeCV
* query sh:property  sh:path/sh:nodeKind/sh:description



In [199]:

sparql_dataset_props_cvs = '''
SELECT ?dataset_prop ?voc_desc
WHERE {
       <http://data.europa.eu/r5r#Dataset_ShapeCV> sh:property ?shape_prop .
       ?shape_prop sh:path ?dataset_prop ;
                   sh:description ?voc_desc  
          
}''' 


ap_dataset_props_CVs = sparql_files_query(query=sparql_dataset_props_cvs, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl'])
for result in ap_dataset_props_CVs:
    print(result)

ap_dataset_props_CVs_list = [item.asdict() for item in ap_dataset_props_CVs]
print(ap_dataset_props_CVs_list)
create_csv(filepath='ap-nl-dataset-CVs.csv',
           headers=['dataset_prop', 'voc_desc'],
           data_dict=ap_dataset_props_CVs_list
           )    

(rdflib.term.URIRef('http://purl.org/dc/terms/accrualPeriodicity'), rdflib.term.Literal('A non EU managed concept is used to indicate the accrualPeriodicity frequency. If no corresponding can be found inform the maintainer of the EU frequency NAL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/language'), rdflib.term.Literal('A non EU managed concept is used to indicate a language. If no corresponding can be found inform the maintainer of the EU language NAL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/publisher'), rdflib.term.Literal('A non EU managed concept is used to indicate the publisher, check if a corresponding exists in the EU corporates bodies NAL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/spatial'), rdflib.term.Literal('A non managed concept is used to indicate a spatial description, check if a corresponding exists'))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#theme'), rdflib.term.Literal('Multiple themes can be used but at least one concept of <http://publica